# CNEFE Data Cleanin

## 0 - Setup

In [1]:
from pyspark.sql import SparkSession
from pathlib import Path
import json
import os

PROJECT_PATH = Path.home() / "projects" / "brazilian_address_linkage"
os.chdir(PROJECT_PATH)

INCOSISTENT_VALS_MAP = json.load(open("config/inconsistent_annotations.json", "r"))
spark = (
    SparkSession.builder.appName("CNEFE Data Cleaning")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.driver.memory", "15g") 
    .config("spark.executor.memory", "5g")
    .getOrCreate()
)
cnefe_raw_df = spark.read.parquet(
    str(PROJECT_PATH / "data/bronze/cnefe/integrated_cnefe_addresses.parquet")
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/08 20:26:49 WARN Utils: Your hostname, lucas-vital-Q570M-D3H, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp8s0)
26/09/08 20:26:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/lucas-vital/projects/brazilian_address_linkage/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/08 20:26:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
cnefe_raw_df.sample(fraction=0.000001, seed=42).toPandas()

/home/lucas-vital/projects/brazilian_address_linkage/.venv/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/08 20:26:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,COD_UNICO_ENDERECO,COD_UF,COD_MUNICIPIO,COD_DISTRITO,COD_SUBDISTRITO,COD_SETOR,NUM_QUADRA,NUM_FACE,CEP,DSC_LOCALIDADE,...,LATITUDE,LONGITUDE,NV_GEO_COORD,COD_ESPECIE,DSC_ESTABELECIMENTO,COD_INDICADOR_ESTAB_ENDERECO,COD_INDICADOR_CONST_ENDERECO,COD_INDICADOR_FINALIDADE_CONST,COD_TIPO_ESPECI,UF
0,60832357,35,3550308,355030872,35503087200,355030872000225P,1,1,03282001,VILA EMA,...,-23.591036,-46.541662,1,1,NaN,NaN,NaN,NaN,103.0,SP
1,220256967,35,3550308,355030875,35503087500,355030875000089P,1,2,08330100,JARDIM VERA CRUZ,...,-23.614342,-46.472248,1,1,NaN,NaN,NaN,NaN,101.0,SP
2,211527897,35,3550308,355030868,35503086800,355030868000106P,2,8,04191140,VILA CARAGUATA,...,-23.647954,-46.606427,1,6,BAR DOS AMIGOS,1,NaN,NaN,NaN,SP
3,225008428,35,3550308,355030819,35503081900,355030819000247P,7,4,05797260,JARDIM IPE,...,-23.654181,-46.772785,1,1,NaN,NaN,NaN,NaN,101.0,SP
4,108427073,29,2910800,291080005,29108000508,291080005080255P,2,1,44023074,RUA NOVA,...,-12.249775,-38.981865,2,1,NaN,NaN,NaN,NaN,103.0,BA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,101275935,35,3514924,351492405,35149240500,351492405000013P,5,4,15823029,CENTRO,...,-21.170534,-49.108529,1,1,NaN,NaN,NaN,NaN,101.0,SP
112,3739154,14,1400100,140010005,14001000500,140010005000223P,1,2,69314032,PISCICULTURA,...,2.836167,-60.734490,1,1,NaN,NaN,NaN,NaN,101.0,RR
113,209291389,43,4316733,431673305,43167330500,431673305000003P,0,132,99952000,VARZEA BONITA,...,-28.163704,-51.850896,1,1,NaN,NaN,NaN,NaN,101.0,RS
114,110758745,17,1703701,170370105,17037010500,170370105000012P,0,52,77560000,BREJINHO,...,-11.270538,-48.526973,1,3,NaN,1,NaN,NaN,NaN,TO


 ## 1-  Nullfy Inconsistent Values 

In [3]:
from pyspark.sql import functions as F
from pyspark.sql import functions as F, DataFrame


def display_missingness(df: DataFrame) -> None:
    total_rows = df.count()

    fulfillment_exprs = [
        F.struct(
            F.lit(c).alias("column"),
            (F.count(F.col(c)) / F.lit(total_rows) * 100).alias("fulfillment_pct"),
        )
        for c in df.columns
    ]

    result = (
        df.select(F.array(*fulfillment_exprs).alias("stats"))
        .selectExpr("explode(stats) as stats")
        .select("stats.column", "stats.fulfillment_pct")
    )

    result.orderBy(F.col("fulfillment_pct").desc()).show(
        len(df.columns), truncate=False
    )


def is_undefined(col_name: str) -> F.Column:
    undefined_expressions = INCOSISTENT_VALS_MAP.get(col_name, [])
    if not undefined_expressions:
        return F.lit(False)
    return F.col(col_name).cast("string").isin(*undefined_expressions)

In [4]:
import pandas as pd

total_rows = cnefe_raw_df.count()
categorical_cols = [
    f.name for f in cnefe_raw_df.schema.fields if str(f.dataType) == "StringType()"
]
undefined_count = []
for col_name in categorical_cols:
    print(f"Column: {col_name}")
    count_undefined_df = (
        cnefe_raw_df.filter(is_undefined(col_name))
        .groupBy(F.col(col_name))
        .count()
        .alias("count")
        .orderBy(F.col("count").desc())
    )

    if count_undefined_df.count() > 0:
        total_undefined_count = count_undefined_df.agg(F.sum("count")).collect()[0][0]
        percentage = round((total_undefined_count / total_rows) * 100, 3)
        undefined_count.append(
            {"column": col_name, "count": total_undefined_count, "pct_db": percentage}
        )

pd.DataFrame(undefined_count)

Column: COD_SETOR
Column: CEP
Column: DSC_LOCALIDADE


Column: NOM_TIPO_SEGLOGR


Column: NOM_TITULO_SEGLOGR
Column: NOM_SEGLOGR


Column: DSC_MODIFICADOR
Column: NOM_COMP_ELEM1
Column: VAL_COMP_ELEM1
Column: NOM_COMP_ELEM2
Column: VAL_COMP_ELEM2
Column: NOM_COMP_ELEM3
Column: VAL_COMP_ELEM3
Column: NOM_COMP_ELEM4
Column: VAL_COMP_ELEM4
Column: NOM_COMP_ELEM5
Column: VAL_COMP_ELEM5
Column: DSC_ESTABELECIMENTO
Column: COD_INDICADOR_ESTAB_ENDERECO
Column: COD_INDICADOR_CONST_ENDERECO
Column: COD_INDICADOR_FINALIDADE_CONST
Column: UF


,column,count,pct_db
0,DSC_LOCALIDADE,7089,0.006
1,NOM_TIPO_SEGLOGR,28041,0.025
2,NOM_SEGLOGR,1236474,1.113


In [5]:
from pyspark.sql import functions as F

columns_names = list(INCOSISTENT_VALS_MAP.keys())

cnefe_cleaned_df = cnefe_raw_df
for c in columns_names:
    cond = is_undefined(c)
    n_bad = cnefe_cleaned_df.filter(cond).count()
    print(f"Detected undefined expressions in column: {c} -> {n_bad}")
    cnefe_cleaned_df = cnefe_cleaned_df.withColumn(
        c, F.when(cond, None).otherwise(F.col(c))
    )

display_missingness(cnefe_cleaned_df.select(columns_names))

Detected undefined expressions in column: DSC_LOCALIDADE -> 7089
Detected undefined expressions in column: CEP -> 0


Detected undefined expressions in column: COD_TIPO_ESPECI -> 0


Detected undefined expressions in column: NOM_SEGLOGR -> 1236474
Detected undefined expressions in column: NOM_TIPO_SEGLOGR -> 28041


+----------------+-----------------+
|column          |fulfillment_pct  |
+----------------+-----------------+
|CEP             |100.0            |
|DSC_LOCALIDADE  |99.99361582677317|
|NOM_TIPO_SEGLOGR|99.97476122917611|
|NOM_SEGLOGR     |98.88709090561338|
|COD_TIPO_ESPECI |81.6401753779999 |
+----------------+-----------------+



## Create complete addres columns

In [6]:
from pyspark.sql import functions as F

address_columns = [
    "NOM_TIPO_SEGLOGR",
    "NOM_TITULO_SEGLOGR",
    "NOM_SEGLOGR",
    "NUM_ENDERECO",
    "DSC_MODIFICADOR",
    "NOM_COMP_ELEM1",
    "VAL_COMP_ELEM1",
    "NOM_COMP_ELEM2",
    "VAL_COMP_ELEM2",
    "NOM_COMP_ELEM3",
    "VAL_COMP_ELEM3",
    "NOM_COMP_ELEM4",
    "VAL_COMP_ELEM4",
    "NOM_COMP_ELEM5",
    "VAL_COMP_ELEM5",
]

cnefe_cleaned_df = cnefe_cleaned_df.withColumn(
    "ENDERECO_COMPLETO",
    F.concat_ws(" ", *[F.col(column) for column in address_columns]),
)

In [7]:
cnefe_cleaned_df.select("ENDERECO_COMPLETO").show(20, truncate=False)

+--------------------------------------------------------+
|ENDERECO_COMPLETO                                       |
+--------------------------------------------------------+
|RUA OSEAS LOPES 59 E                                    |
|RUA GABRIEL MONTEIRO DE CASTRO 26 E ANDAR 1             |
|TRAVESSA 1 GUILHERME PEDROSA 0 SN CASA ANDAR 1          |
|RUA 28 4                                                |
|ACESSO LOCAL 7 26 SMS FUNDOS 2                          |
|RUA VEREADOR ZEZEU RIBEIRO 20 LOJA                      |
|RUA ELISIO MESQUITA 155 E ANDAR 1                       |
|TRAVESSA 2 SALDANHA MARINHO 7                           |
|RUA ENGENHEIRO AGENOR DE FREITAS DE PERIPERI 78 A TERREO|
|RUA DO SABIA 0 SN BLOCO 134 BLOCO A APARTAMENTO 302     |
|CAMINHO 10 0 SN                                         |
|RUA DOM SEBASTIAO LEME 0 SN                             |
|ALAMEDA COLINA DO MAR 1463 A APARTAMENTO 101            |
|ALAMEDA COLINA DO MAR 1579 D APARTAMENTO 202           

## Normalizing Addresses

In [8]:
import re
import unicodedata
from typing import Optional, Dict, List
from pyspark.sql.types import StringType


_ACCENT_SRC = "áàâãäÁÀÂÃÄéèêëÉÈÊËíìîïÍÌÎÏóòôõöÓÒÔÕÖúùûüÚÙÛÜçÇñÑ"
_ACCENT_TGT = "aaaaaAAAAAeeeeEEEEiiiiIIIIoooooOOOOOuuuuUUUUcCnN"


def normalize_address_col(col):
    c = F.regexp_replace(col, r"\p{C}", "")        # strip control chars
    c = F.translate(c, _ACCENT_SRC, _ACCENT_TGT)     # strip accents
    c = F.upper(c)
    c = F.regexp_replace(c, r"[^\w\s]", " ")         # punctuation -> space
    c = F.regexp_replace(c, "_", " ")
    c = F.regexp_replace(c, r"\s+", " ")
    c = F.trim(c)
    return F.when(c == "", None).otherwise(c)

In [9]:
NORMALIZABLE_COLUMNS = [
    "NOM_TIPO_SEGLOGR",
    "NOM_TITULO_SEGLOGR",
    "NOM_SEGLOGR",
    "ENDERECO_COMPLETO",
]

for col_name in NORMALIZABLE_COLUMNS:
    cnefe_cleaned_df = cnefe_cleaned_df.withColumn(
        col_name,
        normalize_address_col(F.col(col_name))
    )

In [10]:
# cnefe_cleaned_df.sample(fraction=0.000001, seed=42).toPandas()

## Criação versão fonetica

In [11]:
from pyspark.sql import functions as F

_PHONETIC_RULES = [
    (r"QU(?=[EI])", "K"), (r"GU(?=[EI])", "G"), (r"QU", "KU"),
    (r"SC(?=[EI])", "S"), (r"XC(?=[EI])", "S"), (r"CH", "X"),
    (r"LH", "LI"), (r"NH", "NI"), (r"PH", "F"),
    (r"C(?=[EI])", "S"), (r"C", "K"), (r"G(?=[EI])", "J"),
    (r"SS", "S"), (r"RR", "R"),
    (r"^H", ""), (r"H", ""), (r"Y", "I"), (r"W", "V"),
]

_LETTER_CLASS = "A-Za-zÀ-ÿ"

# Every accented letter that can appear inside the tokenizer's À-ÿ range,
# mapped to its base letter. Ç/ç -> S/s matches the original's special-case
# (preserves the "soft C" sound instead of collapsing to plain C).
_ACCENT_MAP = {
    "Á":"A","À":"A","Â":"A","Ã":"A","Ä":"A","Å":"A",
    "á":"a","à":"a","â":"a","ã":"a","ä":"a","å":"a",
    "É":"E","È":"E","Ê":"E","Ë":"E","é":"e","è":"e","ê":"e","ë":"e",
    "Í":"I","Ì":"I","Î":"I","Ï":"I","í":"i","ì":"i","î":"i","ï":"i",
    "Ó":"O","Ò":"O","Ô":"O","Õ":"O","Ö":"O","ó":"o","ò":"o","ô":"o","õ":"o","ö":"o",
    "Ú":"U","Ù":"U","Û":"U","Ü":"U","ú":"u","ù":"u","û":"u","ü":"u",
    "Ñ":"N","ñ":"n","Ý":"Y","ý":"y","ÿ":"y",
    "Ç":"S","ç":"s",
}
_ACCENT_FROM = "".join(_ACCENT_MAP.keys())
_ACCENT_TO = "".join(_ACCENT_MAP.values())


def _tokenize_for_phonetic(col):
    """Mirrors re.findall(r"[A-Za-zÀ-ÿ]+|\\d+", text): extract letter/digit
    runs, drop everything else, split at letter<->digit boundaries."""
    c = col
    c = F.regexp_replace(c, f"(?<=[{_LETTER_CLASS}])(?=[0-9])", " ")
    c = F.regexp_replace(c, f"(?<=[0-9])(?=[{_LETTER_CLASS}])", " ")
    c = F.regexp_replace(c, f"[^{_LETTER_CLASS}0-9]+", " ")
    c = F.regexp_replace(c, r"\s+", " ")
    return F.trim(c)


def _letter_token_pipeline(tok):
    """Mirrors _clean_letters + _word_to_phonetic for a single letter token."""
    t = F.translate(tok, _ACCENT_FROM, _ACCENT_TO)
    t = F.upper(t)
    t = F.regexp_replace(t, "[^A-Z]", "")
    for pattern, repl in _PHONETIC_RULES:
        t = F.regexp_replace(t, pattern, repl)
    t = F.regexp_replace(t, r"(.)\1+", r"$1")   # collapse doubled letters
    return t


def generate_phonetic_code_col(col, keep_numbers: bool = True):
    tokenized = _tokenize_for_phonetic(col)
    tokens = F.split(tokenized, " ")

    def _transform_token(tok):
        is_digit = tok.rlike("^[0-9]+$")
        processed = _letter_token_pipeline(tok)
        kept_digit = tok if keep_numbers else F.lit(None)
        return F.when(is_digit, kept_digit).otherwise(processed)

    processed_tokens = F.transform(tokens, _transform_token)
    non_empty = F.filter(processed_tokens, lambda x: x.isNotNull() & (x != ""))
    joined = F.array_join(non_empty, " ")

    return F.when(
        col.isNull() | (F.trim(col) == "") | (F.size(non_empty) == 0),
        F.lit(None),
    ).otherwise(joined)

In [12]:
PHONETICABLE_COLUMNS = [
    "NOM_SEGLOGR",
    "DSC_LOCALIDADE",
    "ENDERECO_COMPLETO",
]

for col_name in PHONETICABLE_COLUMNS:
    cnefe_cleaned_df = cnefe_cleaned_df.withColumn(
        f"{col_name}_phon",
        generate_phonetic_code_col(F.col(col_name))
    )

In [13]:
# cnefe_cleaned_df.sample(fraction=0.0000001, seed=42).toPandas()

## Export cleaned data

In [15]:
EXPORT_FOLDER = PROJECT_PATH / "data/silver/cnefe/"
Path(EXPORT_FOLDER).mkdir(parents=True, exist_ok=True)
cnefe_cleaned_df.write.mode("overwrite").parquet(str(EXPORT_FOLDER / "cleaned_cnefe_addresses.parquet"))